In [ ]:
from pathlib import Path
import os

import numpy as np
from PIL import Image
import torch
from tqdm import tqdm
from rtnls_inference import (
    SegmentationEnsemble,
)
from rtnls_inference.ensembles.predict_output import decollate_predict_full

In [ ]:
ds_path = Path("../samples/fundus")

# these are the output folders for:
av_path = ds_path / "av"                # artery-vein segmentations
vessels_path = ds_path / "vessels"          # optic disc segmentations

device = torch.device("cuda:1")         # device to use for inference

In [ ]:
# Load models
ensemble_av = SegmentationEnsemble.from_huggingface('Eyened/vascx:artery_vein/av_july24.pt').to(device).eval()
ensemble_vessels = (
    SegmentationEnsemble.from_huggingface('Eyened/vascx:vessels/vessels_july24.pt').to(device).eval()
)


In [ ]:
rgb_paths = list((ds_path / 'original').glob('*'))
data = [{"id": path.stem, "image": str(path)} for path in rgb_paths]

In [ ]:
rgb_paths

In [ ]:

# Create dataloader
dataloader = ensemble_av._make_inference_dataloader(
    {"images": data},
    num_workers=8,
    preprocess=True,
    batch_size=8,
)

In [ ]:
# Run inference
av_masks = []
vessel_masks = []
with torch.no_grad():
    for batch in tqdm(dataloader):
        # AV segmentation
        full = ensemble_av.predict_step_full(batch)
        for item in decollate_predict_full(full):
            processed = ensemble_av.postprocess_item(item)
            av_masks.append(processed["output"].astype(np.uint8, copy=False))

        # Vessel segmentation
        full = ensemble_vessels.predict_step_full(batch)
        for item in decollate_predict_full(full):
            processed = ensemble_vessels.postprocess_item(item)
            vessel_masks.append(processed["output"].astype(np.uint8, copy=False))


In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.imshow(av_masks[0])

In [ ]:
plt.imshow(av_masks[1])